## Decision Tree Classification using ID3

As input, accept three possible values - 0, 1, and 2:  
- 0 means using only the pre-pruning approach.  
- 1 means using only the post-pruning approach.  
- 2 means using both types of pruning approaches.

In [1]:
pre_post_both_flag = 2

min_samples = 5
min_gain = 0.01
max_depth = 10
min_improvement = 0.01

Load data:

In [ ]:
%pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
breast_cancer = fetch_ucirepo(id=14)

# data (as pandas dataframes)
X = breast_cancer.data.features
y = breast_cancer.data.targets

# metadata
print(breast_cancer.metadata)

# variable information
print(breast_cancer.variables)

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

In [ ]:
# pd.concat([X, y], axis=1).to_csv("data.csv")

Fill NaN (?)

In [6]:
X

,age,menopause,tumor-size,inv-nodes,node-caps,deg-malig,breast,breast-quad,irradiat
0,30-39,premeno,30-34,0-2,no,3,left,left_low,no
1,40-49,premeno,20-24,0-2,no,2,right,right_up,no
2,40-49,premeno,20-24,0-2,no,2,left,left_low,no
3,60-69,ge40,15-19,0-2,no,2,right,left_up,no
4,40-49,premeno,0-4,0-2,no,2,right,right_low,no
...,...,...,...,...,...,...,...,...,...
281,30-39,premeno,30-34,0-2,no,2,left,left_up,no
282,30-39,premeno,20-24,0-2,no,3,left,left_up,yes
283,60-69,ge40,20-24,0-2,no,1,right,left_up,no
284,40-49,ge40,30-34,5-Mar,no,3,left,left_low,no


In [ ]:
X.apply(lambda col: col.value_counts()).T.stack()  # Python 3.8

age          20-29          1.0
             30-39         36.0
             40-49         90.0
             50-59         96.0
             60-69         57.0
             70-79          6.0
menopause    ge40         129.0
             lt40           7.0
             premeno      150.0
tumor-size   0-4            8.0
             14-Oct        28.0
             15-19         30.0
             20-24         50.0
             25-29         54.0
             30-34         60.0
             35-39         19.0
             40-44         22.0
             45-49          3.0
             50-54          8.0
             9-May          4.0
inv-nodes    0-2          213.0
             11-Sep        10.0
             14-Dec         3.0
             15-17          6.0
             24-26          1.0
             5-Mar         36.0
             8-Jun         17.0
node-caps    no           222.0
             yes           56.0
deg-malig    2            130.0
             3             85.0
        

In [8]:
fix_values_dict = {
    "5-Mar": "3-5",
    "9-May": "5-9",
    "8-Jun": "6-8",
    "11-Sep": "9-11",
    "14-Oct": "10-14",
    "14-Dec": "12-14",
}

X = X.replace(fix_values_dict)
X = X.fillna("unknown")
X

,age,menopause,tumor-size,inv-nodes,node-caps,deg-malig,breast,breast-quad,irradiat
0,30-39,premeno,30-34,0-2,no,3,left,left_low,no
1,40-49,premeno,20-24,0-2,no,2,right,right_up,no
2,40-49,premeno,20-24,0-2,no,2,left,left_low,no
3,60-69,ge40,15-19,0-2,no,2,right,left_up,no
4,40-49,premeno,0-4,0-2,no,2,right,right_low,no
...,...,...,...,...,...,...,...,...,...
281,30-39,premeno,30-34,0-2,no,2,left,left_up,no
282,30-39,premeno,20-24,0-2,no,3,left,left_up,yes
283,60-69,ge40,20-24,0-2,no,1,right,left_up,no
284,40-49,ge40,30-34,3-5,no,3,left,left_low,no


In [9]:
X.apply(lambda col: col.value_counts()).T.stack()

age          20-29          1.0
             30-39         36.0
             40-49         90.0
             50-59         96.0
             60-69         57.0
             70-79          6.0
menopause    ge40         129.0
             lt40           7.0
             premeno      150.0
tumor-size   0-4            8.0
             10-14         28.0
             15-19         30.0
             20-24         50.0
             25-29         54.0
             30-34         60.0
             35-39         19.0
             40-44         22.0
             45-49          3.0
             5-9            4.0
             50-54          8.0
inv-nodes    0-2          213.0
             12-14          3.0
             15-17          6.0
             24-26          1.0
             3-5           36.0
             6-8           17.0
             9-11          10.0
node-caps    no           222.0
             unknown        8.0
             yes           56.0
deg-malig    2            130.0
        

In [10]:
# X.to_csv("data_cleaned.csv")

Stratified 80/20 split:

In [11]:
df_shuffled = pd.concat([X, y], axis=1).sample(frac=1, random_state=1234).reset_index(drop=True)
X = df_shuffled.drop("Class", axis=1)
y = pd.DataFrame(df_shuffled["Class"])

In [12]:
data = pd.concat([X, y], axis=1)

# Stratified sampling
train = data.groupby("Class", group_keys=False).apply(lambda x: x.sample(frac=0.8, random_state=1234))
X_train = train.drop("Class", axis=1)
y_train = pd.DataFrame(train["Class"])

test = data.loc[~data.index.isin(train.index)]
X_test = test.drop("Class", axis=1)
y_test = pd.DataFrame(test["Class"])

In [13]:
print(y.value_counts(normalize=True), len(y))
print(y_train.value_counts(normalize=True), len(y_train))
print(y_test.value_counts(normalize=True), len(y_test))

Class               
no-recurrence-events    0.702797
recurrence-events       0.297203
Name: proportion, dtype: float64 286
Class               
no-recurrence-events    0.703057
recurrence-events       0.296943
Name: proportion, dtype: float64 229
Class               
no-recurrence-events    0.701754
recurrence-events       0.298246
Name: proportion, dtype: float64 57


Decision tree main flow

In [14]:
def calculate_entropy(y):
    n = len(y)
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / n
    entropy = sum(probabilities * -np.log2(probabilities))
    return entropy

In [15]:
calculate_entropy(y)

0.8778446951746506

In [16]:
calculate_entropy(y_train)

0.8775221698372216

In [17]:
calculate_entropy(y[y["Class"] == "no-recurrence-events"])

0.0

In [18]:
def get_information_gain(X, y, feature):
    n = len(y)
    entropy_parent = calculate_entropy(y)
    _, counts = np.unique(X[feature], return_counts=True)
    probabilities = counts / n
    entropy_children = 0
    for value, probability in zip(np.unique(X[feature]), probabilities):
        y_subset = y[X[feature] == value]
        entropy_children += probability * calculate_entropy(y_subset)
    information_gain = entropy_parent - entropy_children
    return information_gain

In [19]:
get_information_gain(X_train, y_train, "age")

0.013375439129579503

In [20]:
get_information_gain(X_train, y_train, "menopause")

0.004663074148666069

In [21]:
get_information_gain(X_train, y_train, "tumor-size")

0.07761941802205807

In [22]:
get_information_gain(X_train, y_train, "deg-malig")

0.09475414061871934

In [23]:
def get_best_feature(X, y):
    information_gains = {feature: get_information_gain(X, y, feature) for feature in X.columns}
    best_feature = max(information_gains, key=information_gains.get)
    return best_feature, information_gains[best_feature]

In [24]:
get_best_feature(X, y)

('deg-malig', 0.07700985251661441)

In [25]:
class Node:
    def __init__(self, feature=None, value=None, label=None, branches=None):
        self.feature = feature
        self.value = value
        self.label = label
        self.branches = branches if branches else {}

In [26]:
def get_majority_class(y):
    if len(y) == 0:
        return None
    return y.value_counts().idxmax()[0]

In [27]:
get_majority_class(y_train)

'no-recurrence-events'

In [28]:
def should_pre_prune(X, y, best_feature_value, min_samples, min_gain):
    if len(y) < min_samples:
        return True

    return best_feature_value < min_gain

In [29]:
def build_tree(X, y, pre_pruning=True, depth=0, min_samples=min_samples, min_gain=min_gain, max_depth=max_depth):
    if len(y) == 0:
        print("No samples left.")
        return Node(label=get_majority_class(y))

    if y.nunique()[0] == 1:
        # print("Only one class left.")
        return Node(label=get_majority_class(y))

    if X.empty or depth >= max_depth:
        print("Max depth reached.")
        return Node(label=get_majority_class(y))

    best_feature, best_feature_value = get_best_feature(X, y)

    if pre_pruning and should_pre_prune(X, y, best_feature_value, min_samples, min_gain):
        # print("Pre-pruning.")
        return Node(label=get_majority_class(y))

    node = Node(feature=best_feature)

    for value in X[best_feature].unique():
        mask = X[best_feature] == value
        if not mask.any():
            continue

        X_subset = X[mask].drop(columns=[best_feature])
        y_subset = y[mask]

        child = build_tree(
            X_subset,
            y_subset,
            depth=depth + 1,
            pre_pruning=pre_pruning,
            min_samples=min_samples,
            min_gain=min_gain,
            max_depth=max_depth,
        )
        child.parent = node
        node.branches[value] = child

    if not node.branches:
        return Node(label=get_majority_class(y))

    return node

In [30]:
root = build_tree(X_train, y_train)

In [31]:
def print_tree(node, depth=0):
    indent = "  " * depth

    if node.label is not None:
        print(f"{indent}Leaf: {node.label}")
        return

    print(f"{indent}Feature: {node.feature}")

    for value, child in node.branches.items():
        print(f"{indent}  If {node.feature} == {value}:")
        print_tree(child, depth + 1)

In [32]:
print_tree(root)

Feature: deg-malig
  If deg-malig == 2:
  Feature: tumor-size
    If tumor-size == 45-49:
    Leaf: no-recurrence-events
    If tumor-size == 35-39:
    Leaf: no-recurrence-events
    If tumor-size == 15-19:
    Feature: menopause
      If menopause == ge40:
      Leaf: no-recurrence-events
      If menopause == premeno:
      Feature: breast
        If breast == right:
        Leaf: no-recurrence-events
        If breast == left:
        Leaf: recurrence-events
      If menopause == lt40:
      Leaf: no-recurrence-events
    If tumor-size == 20-24:
    Feature: breast-quad
      If breast-quad == central:
      Leaf: no-recurrence-events
      If breast-quad == left_up:
      Feature: menopause
        If menopause == premeno:
        Leaf: no-recurrence-events
        If menopause == ge40:
        Feature: age
          If age == 60-69:
          Leaf: no-recurrence-events
          If age == 50-59:
          Leaf: no-recurrence-events
          If age == 40-49:
          Leaf: recur

In [33]:
def predict_single(node, sample):
    if node.label is not None:
        return node.label

    feature_value = sample.get(node.feature)

    if feature_value not in node.branches:
        first_branch_label = next(iter(node.branches.values())).label
        return first_branch_label

    return predict_single(node.branches[feature_value], sample)

In [34]:
def predict(root, X_test, y_test):
    predictions = X_test.apply(lambda sample: predict_single(root, sample), axis=1)
    accuracy = np.mean(predictions == y_test["Class"]) * 100
    return accuracy, predictions

In [35]:
def post_prune(node, X_test, y_test, min_improvement=min_improvement):
    if node.label is not None:
        return

    current_accuracy, _ = predict(node, X_test, y_test)

    pruned_node = deepcopy(node)
    pruned_node.label = get_majority_class(y_test)
    pruned_node.branches = {}

    pruned_accuracy, _ = predict(pruned_node, X_test, y_test)

    if pruned_accuracy >= current_accuracy + min_improvement:
        node.label = pruned_node.label
        node.branches = {}
        return

    for child in node.branches.values():
        post_prune(child, X_test, y_test, min_improvement)

In [36]:
def id3(
    X_train,
    y_train,
    X_test,
    y_test,
    pre_post_both_flag=pre_post_both_flag,
    min_samples=min_samples,
    min_gain=min_gain,
    max_depth=max_depth,
):

    pre_pruning = False if pre_post_both_flag == 1 else True
    post_pruning = False if pre_post_both_flag == 0 else True

    root = build_tree(
        X_train,
        y_train,
        depth=0,
        pre_pruning=pre_pruning,
        min_samples=min_samples,
        min_gain=min_gain,
        max_depth=max_depth,
    )

    if post_pruning:
        post_prune(root, X_test, y_test)

    return predict(root, X_test, y_test)

In [37]:
accuracy, predictions = id3(X_train, y_train, X_test, y_test)
accuracy

70.17543859649122

Cross validation

In [38]:
def get_cross_validation_data(X, y, group_number=1, cross_groups=10):
    group_len = len(X) // cross_groups

    indices = np.arange(len(X))

    start = (group_number - 1) * group_len
    end = (group_number) * group_len
    mask = (indices >= start) & (indices < end)

    return X[~mask], y[~mask], X[mask], y[mask]

In [39]:
def one_fold_experiment(X, y, group_number, cross_groups):
    X_train, y_train, X_test, y_test = get_cross_validation_data(
        X, y, group_number=group_number, cross_groups=cross_groups
    )
    accuracy, predictions = id3(X_train, y_train, X_test, y_test)
    print(f"Accuracy Fold {group_number}: {accuracy:2f}%")

    return accuracy

In [40]:
def cross_validation_experiments(X, y, cross_groups):
    accuracies = []
    for group_number in range(1, cross_groups + 1):
        accuracy = one_fold_experiment(X, y, group_number, cross_groups)
        accuracies.append(accuracy)

    average_accuracy = np.mean(accuracies)
    standard_deviation = np.std(accuracies)
    return average_accuracy, standard_deviation

In [41]:
def experiment(cross_groups=10):
    train_accuracy, train_predictions = id3(X_train, y_train, X_train, y_train)
    print(f"1. Train set accuracy: \nAccuracy: {train_accuracy:2f}%\n")

    print(f"2. {cross_groups}-Fold Cross-Validation Results:")
    average_accuracy, standard_deviation = cross_validation_experiments(X, y, cross_groups)
    print(f"\nAverage Accuracy: {average_accuracy:2f}%\nStandard Deviation: {standard_deviation:2f}\n")

    test_accuracy, test_predictions = id3(X_train, y_train, X_test, y_test)
    print(f"3. Test set accuracy: \nAccuracy: {test_accuracy:2f}%\n")

    # return train_accuracy, average_accuracy, standard_deviation, test_accuracy

In [42]:
experiment()

1. Train set accuracy: 
Accuracy: 75.545852%

2. 10-Fold Cross-Validation Results:
Accuracy Fold 1: 75.000000%
Accuracy Fold 2: 78.571429%
Accuracy Fold 3: 71.428571%
Accuracy Fold 4: 64.285714%
Accuracy Fold 5: 53.571429%
Accuracy Fold 6: 71.428571%
Accuracy Fold 7: 71.428571%
Accuracy Fold 8: 82.142857%
Accuracy Fold 9: 75.000000%
Accuracy Fold 10: 60.714286%

Average Accuracy: 70.357143%
Standard Deviation: 8.151937

3. Test set accuracy: 
Accuracy: 70.175439%

